# Python Main Notebook (MATLAB `main_sub.m` Equivalent)

이 노트북은 MATLAB `src/main_sub.m`의 역할을 Python에서 셀 단위로 실행하도록 구성했습니다.

실행 순서:
1. 환경/모듈 임포트
2. 설정(`initialize_config`) 구성
3. H5 데이터 로딩
4. 단일 필터 디버그 실행
5. 전체 메인 루프 실행 (`main_sub.m` 대응)
6. 결과 CSV/시각화 확인

In [ ]:
# Cell 1: Environment setup
%matplotlib inline

from pathlib import Path
import importlib
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
np.random.seed(SEED)

ROOT = Path('..').resolve()
PY_DIR = Path('.').resolve()
print('ROOT =', ROOT)
print('PY_DIR =', PY_DIR)

In [ ]:
# Cell 2: Import Python pipeline modules
import config as config_module
import data_loader as data_loader_module
import runner as runner_module

importlib.reload(config_module)
importlib.reload(data_loader_module)
importlib.reload(runner_module)

from config import initialize_config
from data_loader import load_simulation_data
from runner import FILTER_NAMES, run_filter, run_main_like

print('Imported modules successfully.')
print('Filters:', FILTER_NAMES)

In [ ]:
# Cell 3: Build config (MATLAB initializeConfig 대응)
cfg = initialize_config()

# Optional overrides for quick debugging
cfg.path_data = '../data'
cfg.path_result = '../result'
cfg.motion_model = 'cv'  # 'cv' or 'imm'
cfg.iterations = 1000

print(cfg)

In [ ]:
# Cell 4: Load H5 data (MATLAB loadSimulationData 대응)
h5_name = 'simulation_data_imm.h5' if cfg.motion_model.lower() == 'imm' else 'simulation_data.h5'
h5_path = str(Path(cfg.path_data) / h5_name)

data = load_simulation_data(h5_path)

print('Loaded:', h5_path)
print('ranging:', data.ranging.shape)
print('x_hat_LLS:', data.x_hat_lls.shape)
print('z_LLS:', data.z_lls.shape)
print('R_LLS:', data.r_lls.shape)
print('true_state:', data.true_state.shape)

In [ ]:
# Cell 5: Single-filter debug run (MATLAB runFilter 대응 일부)
# Fast sanity check before full benchmark
cfg_debug = initialize_config(num_particles=100)
cfg_debug.path_data = cfg.path_data
cfg_debug.path_result = cfg.path_result
cfg_debug.motion_model = cfg.motion_model
cfg_debug.iterations = min(cfg.iterations, data.x_hat_lls.shape[2])

_, metric_debug = run_filter('NonlinearParticleFilter', data, cfg_debug)
print('Debug RMSE by noise:', metric_debug['RMSE'])
print('Debug APE by noise:', metric_debug['APE'])

In [ ]:
# Cell 6: Main benchmark run (MATLAB main_sub 대응)
# Warning: this can take time (all filters x all particleCounts x all noise levels)

cfg_main = initialize_config()
cfg_main.path_data = cfg.path_data
cfg_main.path_result = cfg.path_result
cfg_main.motion_model = cfg.motion_model
cfg_main.iterations = cfg.iterations

all_rmse_csv_path = run_main_like(cfg_main)
print('Saved aggregated CSV:', all_rmse_csv_path)

In [ ]:
# Cell 7: Inspect aggregated results
result_df = pd.read_csv(all_rmse_csv_path)
print(result_df.head())
print('Rows:', len(result_df), 'Columns:', len(result_df.columns))

# Quick visualization for RMSE rows only
if 'RowType' in result_df.columns:
    rmse_only = result_df[result_df['RowType'] == 'RMSE'].copy()
else:
    rmse_only = result_df.copy()

if 'NoiseVariance' in rmse_only.columns and 'NonlinearParticleFilter' in rmse_only.columns:
    plt.figure(figsize=(6, 4))
    plt.semilogx(rmse_only['NoiseVariance'], rmse_only['NonlinearParticleFilter'], marker='o')
    plt.xlabel('NoiseVariance')
    plt.ylabel('RMSE')
    plt.title('NonlinearParticleFilter RMSE')
    plt.grid(True)
    plt.show()
else:
    print('Expected columns not found for quick plot.')